## Experimento 1:
1. Testando o parâmetro weight na função CrossEntropyLoss
2. Obs.: Observar resultados dos belgian_blocks

In [2]:
# Importação das bibliotecas e inicialização dos dados
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # Onde o programa vai ser executado, se existir uma gpu compatível com cuda (nvidia) disponível, usa a GPU (mais rapido), se não vai para cpu

train_path = "dataset_processed/train"
test_path  = "dataset_processed/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

cpu


In [3]:
# Dados de Treino
train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print(classes)

['asphalt', 'belgian_blocks', 'offroad']


In [4]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    pesos = torch.tensor([1.0, 4.0, 2.5], dtype=torch.float32).to(device) # MUDANÇA

    criterion = nn.CrossEntropyLoss(weight=pesos) # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.001) # atualiza os pesos da rede para reduzir o erro

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [5]:
# Função Teste
def evaluate_model(model):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images) # gera as previsões
            _, preds = torch.max(outputs, 1) # escolhe a classe com o maior valor

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred) # calcula a porcentaggem de acertos

    print("Accuracy:", accuracy)
    print(classification_report(y_true, y_pred, target_names=classes)) # mostra as métricas por classe

In [6]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 12.4617
Epoch 2/10 Loss: 5.1087
Epoch 3/10 Loss: 5.5839
Epoch 4/10 Loss: 3.5148
Epoch 5/10 Loss: 4.8567
Epoch 6/10 Loss: 4.0306
Epoch 7/10 Loss: 1.8327
Epoch 8/10 Loss: 5.3322
Epoch 9/10 Loss: 2.9847
Epoch 10/10 Loss: 2.2315
Tempo treino: 524.64 segundos
Accuracy: 0.9266666666666666
                precision    recall  f1-score   support

       asphalt       0.94      0.99      0.97       218
belgian_blocks       0.88      0.44      0.58        32
       offroad       0.87      0.96      0.91        50

      accuracy                           0.93       300
     macro avg       0.90      0.80      0.82       300
  weighted avg       0.92      0.93      0.92       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 7.6306
Epoch 2/10 Loss: 4.8338
Epoch 3/10 Loss: 1.6491
Epoch 4/10 Loss: 1.6959
Epoch 5/10 Loss: 2.3810
Epoch 6/10 Loss: 0.8078
Epoch 7/10 Loss: 2.9525
Epoch 8/10 Loss: 2.7251
Epoch 9/10 Loss: 5.3488
Epoch 10/10 Loss: 3.3755
Tempo treino